# Continue notebook 31 pilots: simple, then tactical

This notebook reloads the ten `best_model.pt` checkpoints produced by notebook 31 and continues each pilot for two separate 200-epoch populations. The first population uses the same `simple` scripted opponent as notebook 31; the second starts from the selected checkpoints of that population and uses the `tactical` scripted opponent.

Each phase uses 60 training scenarios per epoch, 60 validation scenarios for checkpoint selection, and 60 post-selection test scenarios. Every training/evaluation flight is limited to 300 in-game seconds and terminates as soon as either aircraft is eliminated. Training scenarios are resampled each epoch. The second phase is therefore a tactical continuation, not a fresh random initialization. Each continuation creates a fresh Adam optimizer while retaining the source policy and critic.

After the tactical phase, the final population is passed to the 1-v-1 NN-versus-NN forecasting-corpus collector. The collector runs both aircraft with trained policies; it does not use the scripted opponent during corpus collection.

## Before running

Run from the repository root with the ML/video environment installed. Set `BVR_PILOT31_POPULATION` if automatic discovery does not select the completed notebook 31 population. The default run ID is a four-digit sequential directory (`0001`, `0002`, ...), not a timestamp.

Useful controls are `BVR_CONTINUATION_ID`, `BVR_CONTINUATION_ROOT`, `BVR_CONTINUATION_WORKERS`, `BVR_CONTINUATION_DEVICE`, `BVR_CONTINUATION_EPISODE_SECONDS`, and `BVR_CORPUS_WORKERS`. A 240-character absolute-path budget is checked before training and corpus generation to leave margin below the Windows legacy 260-character limit.

In [ ]:
import json
import os
import re
import subprocess
import sys
from dataclasses import fields, replace
from pathlib import Path

import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'bvr_behavior_prediction').exists():
    REPO_ROOT = REPO_ROOT.parent
for source_root in (REPO_ROOT, REPO_ROOT / 'bvr_sim_source'):
    if str(source_root) not in sys.path:
        sys.path.insert(0, str(source_root))

from bvr_behavior_prediction.rl.bvr_jsbsim_backend import BVRJSBSimBackend
from bvr_behavior_prediction.rl.config import PilotTrainingConfig, RewardWeights
from bvr_behavior_prediction.rl.dogfight_reward import (
    DogfightWeights, hybrid_dogfight_reward_definition,
)
from bvr_behavior_prediction.rl.environment import BluePilotEnvironment
from bvr_behavior_prediction.rl.multi_pilot import PilotSpec, train_pilots
from bvr_behavior_prediction.rl.observation_features import RADAR_POLICY_OBSERVATION_SIZE

EPOCHS = int(os.getenv('BVR_CONTINUATION_EPOCHS', '200'))
SCENARIOS = int(os.getenv('BVR_CONTINUATION_SCENARIOS', '60'))
EPISODE_SECONDS = float(os.getenv('BVR_CONTINUATION_EPISODE_SECONDS', '300'))
PILOT_COUNT = 10
DEVICE = os.getenv('BVR_CONTINUATION_DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu')
DEFAULT_WORKERS = max(1, ((os.cpu_count() or 1) * 80) // 100)
SIMULATOR_WORKERS = int(os.getenv('BVR_CONTINUATION_WORKERS', str(DEFAULT_WORKERS)))
SIMULATOR_EXECUTOR = os.getenv('BVR_CONTINUATION_EXECUTOR', 'process')
MAX_WINDOWS_PATH = 240

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf8'))

def assert_path_budget(*paths):
    for path in paths:
        absolute = str(Path(path).resolve())
        if len(absolute) > MAX_WINDOWS_PATH:
            raise ValueError(
                f'Windows-safe path budget exceeded ({len(absolute)} > {MAX_WINDOWS_PATH}): {absolute}'
            )

def next_short_id(parent):
    parent = Path(parent)
    parent.mkdir(parents=True, exist_ok=True)
    requested = os.getenv('BVR_CONTINUATION_ID')
    if requested is not None:
        if not requested.isdigit() or not 1 <= int(requested) <= 9999:
            raise ValueError('BVR_CONTINUATION_ID must be between 0001 and 9999')
        return f'{int(requested):04d}'
    used = [int(path.name) for path in parent.iterdir()
            if path.is_dir() and re.fullmatch(r'\d{4}', path.name)]
    next_value = max(used, default=0) + 1
    if next_value > 9999:
        raise RuntimeError('No four-digit continuation IDs remain')
    return f'{next_value:04d}'

def discover_source_population():
    configured = os.getenv('BVR_PILOT31_POPULATION')
    if configured:
        return Path(configured).expanduser().resolve()
    root = REPO_ROOT / 'artifacts' / 'rl_pilot_notebook' / 'hybrid'
    candidates = [path.parent for path in root.glob('*/pilots.json')]
    if not candidates:
        raise FileNotFoundError(
            'No completed notebook 31 population found; set BVR_PILOT31_POPULATION'
        )
    return max(candidates, key=lambda path: path.stat().st_mtime).resolve()

SOURCE_POPULATION = discover_source_population()
CONTINUATION_ROOT = Path(os.getenv(
    'BVR_CONTINUATION_ROOT', REPO_ROOT / 'artifacts' / 'rl_pilot_notebook' / 'continuations'
))
assert_path_budget(CONTINUATION_ROOT)
RUN_ID = next_short_id(CONTINUATION_ROOT)
RUN_ROOT = (CONTINUATION_ROOT / RUN_ID).resolve()
if RUN_ROOT.exists():
    raise FileExistsError(f'Continuation output already exists: {RUN_ROOT}')
RUN_ROOT.mkdir(parents=True)
SIMPLE_OUTPUT = RUN_ROOT / 'simple'
TACTICAL_OUTPUT = RUN_ROOT / 'tactical'
CORPUS_OUTPUT = (REPO_ROOT / 'shared' / 'corpora' / 'nn-duels-current' / RUN_ID).resolve()
assert_path_budget(
    SOURCE_POPULATION, RUN_ROOT, SIMPLE_OUTPUT / 'pilot_001' / 'simulator_logs',
    TACTICAL_OUTPUT / 'pilot_001' / 'simulator_logs',
    CORPUS_OUTPUT / 'A01' / 'observations' / 'nn_26110600_00000.parquet',
)
if EPOCHS < 1 or SCENARIOS < 3 or SIMULATOR_WORKERS < 0:
    raise ValueError('epochs/scenarios/workers are outside their supported ranges')
if not 0 < EPISODE_SECONDS <= 300:
    raise ValueError('BVR_CONTINUATION_EPISODE_SECONDS must be in (0, 300]')
if SIMULATOR_EXECUTOR not in ('thread', 'process'):
    raise ValueError('BVR_CONTINUATION_EXECUTOR must be thread or process')
print({
    'source_population': str(SOURCE_POPULATION), 'run_id': RUN_ID,
    'device': DEVICE, 'epochs_per_phase': EPOCHS, 'scenarios_per_role': SCENARIOS,
    'episode_seconds': EPISODE_SECONDS,
    'simulator_workers': SIMULATOR_WORKERS, 'simulator_executor': SIMULATOR_EXECUTOR,
    'simple_output': str(SIMPLE_OUTPUT), 'tactical_output': str(TACTICAL_OUTPUT),
    'corpus_output': str(CORPUS_OUTPUT),
})

## Reload the notebook 31 population

The parent manifest is authoritative for pilot IDs, reward definitions, seeds, and sampled architectures. `initialize_from` in `train_pilots` verifies each source checkpoint hash, loads both policy and critic, and retains that loaded checkpoint as the initial validation candidate.

In [ ]:
def reward_from_manifest(serialized):
    if serialized['version'] != '1':
        raise ValueError(f"Unsupported notebook 31 reward version: {serialized['version']}")
    parameters = dict(serialized['parameters'])
    weights = DogfightWeights(**parameters.pop('weights'))
    return hybrid_dogfight_reward_definition(
        serialized['name'], weights=weights, **parameters
    )

parent_manifest = read_json(SOURCE_POPULATION / 'pilots.json')
parent_records = parent_manifest['pilots']
if len(parent_records) != PILOT_COUNT or any(p['status'] != 'completed' for p in parent_records):
    raise ValueError('Notebook 31 source must contain exactly ten completed pilots')

source_metadata = []
PILOTS = []
for record in parent_records:
    run_dir = SOURCE_POPULATION / record['output_dir']
    metadata = read_json(run_dir / 'config.json')
    source_metadata.append(metadata)
    PILOTS.append(PilotSpec(
        pilot_id=record['pilot_id'], seed=int(record['seed']),
        reward=reward_from_manifest(record['reward']),
        training_overrides=dict(record.get('training_overrides', {})),
    ))

OBSERVATION_SIZE = source_metadata[0]['observation_size']
if OBSERVATION_SIZE != RADAR_POLICY_OBSERVATION_SIZE or OBSERVATION_SIZE != 114:
    raise ValueError(f'Expected notebook 31 114-feature pilots, got {OBSERVATION_SIZE}')
if any(metadata['observation_size'] != OBSERVATION_SIZE for metadata in source_metadata):
    raise ValueError('Source pilots do not share one observation contract')

def source_config_settings():
    training = dict(source_metadata[0]['training'])
    known = {field.name for field in fields(PilotTrainingConfig)}
    settings = {key: value for key, value in training.items() if key in known}
    settings['reward'] = RewardWeights(**settings['reward'])
    settings['reuse_simulator_environments'] = True
    return settings

BENCHMARK_REWARD = reward_from_manifest(parent_manifest['benchmark_reward'])
print(f'Loaded {len(PILOTS)} pilots from {SOURCE_POPULATION}')
display(pd.DataFrame([{
    'pilot_id': pilot.pilot_id, 'seed': pilot.seed,
    **pilot.training_overrides, 'reward': pilot.reward.name,
} for pilot in PILOTS]))

## Configure the two continuation phases

The phase seed ranges are disjoint. Training is resampled into a fresh 60-scenario batch at every epoch; validation and test each use their own fixed 60-scenario suite. Pilot IDs and reward profiles stay unchanged so checkpoint initialization can be verified between phases.

In [ ]:
def make_environment_factory(opponent_type):
    if opponent_type not in ('simple', 'tactical'):
        raise ValueError('opponent_type must be simple or tactical')

    def builder(run_config):
        def backend_factory(scenario_dict, recording_path=None):
            return BVRJSBSimBackend(
                scenario_dict, recording_path=recording_path,
                log_dir=run_config.output_dir / 'simulator_logs',
                opponent_type=opponent_type, include_hostile_missiles=True,
                include_radar_features=True, episode_duration_s=run_config.episode_duration_s,
                terminate_on_aircraft_elimination=True,
            )

        def environment_factory(scenario, recording_path=None):
            return BluePilotEnvironment(
                backend_factory, scenario,
                planning_horizon_s=run_config.planning_horizon_s,
                episode_duration_s=run_config.episode_duration_s,
                history_duration_s=run_config.history_duration_s,
                sample_interval_s=run_config.sample_interval_s,
                recording_path=recording_path,
            )

        return environment_factory
    return builder

PHASE_SEEDS = {
    'simple': {'training': 1_000_000, 'validation': 2_000_000, 'test': 3_000_000},
    'tactical': {'training': 4_000_000, 'validation': 5_000_000, 'test': 6_000_000},
}

def phase_pilots(phase):
    offset = PHASE_SEEDS[phase]['training']
    return [replace(pilot, seed=pilot.seed + offset) for pilot in PILOTS]

def phase_config(phase, output_dir):
    seeds = PHASE_SEEDS[phase]
    settings = source_config_settings()
    settings.update({
        'epochs': EPOCHS, 'scenarios_per_epoch': SCENARIOS,
        'episode_duration_s': EPISODE_SECONDS,
        'evaluation_scenarios_per_epoch': SCENARIOS,
        'test_scenarios_per_pilot': SCENARIOS,
        'resample_training_scenarios': True,
        'evaluation_seed': seeds['validation'], 'test_seed': seeds['test'],
        'simulator_workers': SIMULATOR_WORKERS, 'simulator_executor': SIMULATOR_EXECUTOR,
        'output_dir': Path(output_dir), 'checkpoint_interval': max(1, EPOCHS // 10),
        'demonstration_interval': 0, 'diagnostic_interval': 1,
        'mlflow_experiment': f'jsbsim-notebook31-continuation-{phase}',
        'mlflow_tracking_uri': f"sqlite:///{(RUN_ROOT / f'{phase}.db').as_posix()}",
    })
    return PilotTrainingConfig(**settings)

simple_config = phase_config('simple', SIMPLE_OUTPUT)
tactical_config = phase_config('tactical', TACTICAL_OUTPUT)
print(json.dumps({
    'simple': simple_config.as_dict(), 'tactical': tactical_config.as_dict(),
}, indent=2, default=str))

## Phase 1: continue against the simple opponent

Every pilot is initialized from its own notebook 31 best checkpoint. The source population is not modified.

In [ ]:
simple_results = train_pilots(
    pilots=phase_pilots('simple'), config=simple_config,
    env_factory_builder=make_environment_factory('simple'),
    observation_size=OBSERVATION_SIZE, device=DEVICE,
    benchmark_reward=BENCHMARK_REWARD,
    population_metadata={
        'experiment': 'notebook31_checkpoint_continuation',
        'phase': 'simple', 'opponent_type': 'simple',
        'episode_termination': 'aircraft elimination or time limit',
        'terminate_on_aircraft_elimination': True,
        'episode_duration_s': EPISODE_SECONDS,
        'source_population': str(SOURCE_POPULATION), 'run_id': RUN_ID,
        'epochs': EPOCHS, 'training_scenarios_per_epoch': SCENARIOS,
        'validation_scenarios': SCENARIOS, 'test_scenarios': SCENARIOS,
    },
    initialize_from=SOURCE_POPULATION,
)
simple_summary = pd.DataFrame(simple_results)
display(simple_summary[['pilot_id', 'selection_return', 'benchmark_return', 'checkpoint']])

## Phase 2: continue against the tactical opponent

The tactical phase starts from the best checkpoints selected by phase 1. It uses the same ten reward profiles and architectures, but a fresh seed range and the tactical scripted opponent.

In [ ]:
tactical_results = train_pilots(
    pilots=phase_pilots('tactical'), config=tactical_config,
    env_factory_builder=make_environment_factory('tactical'),
    observation_size=OBSERVATION_SIZE, device=DEVICE,
    benchmark_reward=BENCHMARK_REWARD,
    population_metadata={
        'experiment': 'notebook31_checkpoint_continuation',
        'phase': 'tactical', 'opponent_type': 'tactical',
        'episode_termination': 'aircraft elimination or time limit',
        'terminate_on_aircraft_elimination': True,
        'episode_duration_s': EPISODE_SECONDS,
        'source_population': str(SIMPLE_OUTPUT), 'run_id': RUN_ID,
        'epochs': EPOCHS, 'training_scenarios_per_epoch': SCENARIOS,
        'validation_scenarios': SCENARIOS, 'test_scenarios': SCENARIOS,
    },
    initialize_from=SIMPLE_OUTPUT,
)
tactical_summary = pd.DataFrame(tactical_results)
display(tactical_summary[['pilot_id', 'selection_return', 'benchmark_return', 'checkpoint']])

## Generate the 1-v-1 forecasting corpus

The collector uses the completed tactical population and controls both aircraft. Each flight is capped at 300 seconds and the collector breaks immediately after either aircraft is eliminated. Notebook 31's failure was in the simulator reset diagnostic: self-play sets `blue_opponent_type=None`, leaving the blue ego aircraft without a scripted baseline. `_gen_expert_action()` previously inserted `None` into a NumPy array and raised `ValueError: setting an array element with a sequence`. The simulator now emits a zero expert-action diagnostic for NN-controlled ego aircraft; this does not replace or alter the actions passed to `env.step()`.

The subprocess output is captured and printed so a future collector failure includes its real child traceback rather than only a generic `CalledProcessError`.

In [ ]:
CORPUS_FLIGHTS = int(os.getenv('BVR_CORPUS_FLIGHTS', '10000'))
CORPUS_SEED = int(os.getenv('BVR_CORPUS_SEED', '26110600'))
CORPUS_EPISODE_SECONDS = float(os.getenv('BVR_CORPUS_EPISODE_SECONDS', str(EPISODE_SECONDS)))
CORPUS_WORKERS = int(os.getenv('BVR_CORPUS_WORKERS', '0'))
if CORPUS_FLIGHTS < 3 or CORPUS_WORKERS < 0:
    raise ValueError('corpus flights must be at least 3 and workers nonnegative')
if not 0 < CORPUS_EPISODE_SECONDS <= 300:
    raise ValueError('BVR_CORPUS_EPISODE_SECONDS must be in (0, 300]')
assert_path_budget(CORPUS_OUTPUT / 'A01' / 'observations' / f'nn_{CORPUS_SEED}_00000.parquet')
collector = REPO_ROOT / 'scripts' / 'generate_forecasting_corpus.py'
command = [
    sys.executable, str(collector), '--population', str(TACTICAL_OUTPUT),
    '--output', str(CORPUS_OUTPUT), '--flights', str(CORPUS_FLIGHTS),
    '--seed', str(CORPUS_SEED), '--episode-seconds', str(CORPUS_EPISODE_SECONDS),
    '--workers', str(CORPUS_WORKERS),
]
completed = subprocess.run(command, capture_output=True, text=True)
if completed.stdout:
    print(completed.stdout, end='')
if completed.returncode:
    if completed.stderr:
        print(completed.stderr, end='', file=sys.stderr)
    raise RuntimeError(f'Corpus collector failed with exit code {completed.returncode}: {command!r}')
corpus_manifest = read_json(CORPUS_OUTPUT / 'manifest.json')
corpus_schema = read_json(CORPUS_OUTPUT / 'schema.json')
assert corpus_manifest['observation_size'] == OBSERVATION_SIZE == 114
assert corpus_manifest['physical_flights'] == CORPUS_FLIGHTS
assert corpus_manifest['perspective_episodes'] == 2 * CORPUS_FLIGHTS
assert len(corpus_schema['observation_columns']) == OBSERVATION_SIZE
print(json.dumps({
    'corpus': str(CORPUS_OUTPUT), 'physical_flights': corpus_manifest['physical_flights'],
    'perspective_episodes': corpus_manifest['perspective_episodes'],
    'observation_size': corpus_manifest['observation_size'],
}, indent=2))

## Completion record

The final artifacts are under the short `RUN_ID` directory. The tactical `pilots.json` is the population used for corpus generation; its per-pilot manifests contain checkpoint hashes, initialization records, validation selection, and 60-case test results.